In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats


## 1. Introduction

This project uses the Spotify Music Tracks dataset. Each row represents one Spotify track and includes metadata such as the track name, artist name, popularity score, duration, release date, explicit status, and genre. The dataset also includes audio features such as danceability, energy, valence, acousticness, instrumentalness, loudness, tempo, and speechiness.

The main question we want to explore is:

**Can we predict whether a Spotify track becomes popular using its audio features, genre and track metadata?**

This question helps us better understand how audio characteristics and certain genres are associated with successful songs, providing insights into modern music trends and listener preferences. 

In [2]:
tracks = pd.read_csv("data/music_tracks.csv")

In [3]:
tracks.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,NaN,4,acoustic


In [4]:
tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   release_date      114000 non-null  object 
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000 non-null  int64  
 14  speechiness       114000 non-null  float64
 15  acousticness      114000 non-null  float64
 16  instrumentalness  11

The dataset, `music_tracks.csv`, contains 114,000 rows and 22 columns. Each row represents one Spotify track with audio features, metadata, popularity, and genre information.

In [5]:
print("Tracks dataset shape:", tracks.shape)

Tracks dataset shape: (114000, 22)


In [6]:
tracks.describe()

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.000000,114000.000000,1.140000e+05,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,91886.000000,114000.000000
mean,56999.500000,33.238535,2.280292e+05,0.566800,0.641383,5.309140,-8.258960,0.637553,0.084652,0.314910,0.156050,0.213553,0.474068,123.119961,3.904035
std,32909.109681,22.305078,1.072977e+05,0.173542,0.251529,3.559987,5.029337,0.480709,0.105732,0.332523,0.309555,0.190378,0.259261,29.784235,0.432621
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,28499.750000,17.000000,1.740660e+05,0.456000,0.472000,2.000000,-10.013000,0.000000,0.035900,0.016900,0.000000,0.098000,0.260000,99.999000,4.000000
50%,56999.500000,35.000000,2.129060e+05,0.580000,0.685000,5.000000,-7.004000,1.000000,0.048900,0.169000,0.000042,0.132000,0.464000,122.998000,4.000000
75%,85499.250000,50.000000,2.615060e+05,0.695000,0.854000,8.000000,-5.003000,1.000000,0.084500,0.598000,0.049000,0.273000,0.683000,141.649000,4.000000
max,113999.000000,100.000000,5.237295e+06,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,222.605000,5.000000


In [7]:
tracks.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'release_date', 'explicit', 'danceability',
       'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')

## 2. Data Cleaning and EDA


In [8]:
cleaned = tracks.copy()

In [9]:
# keep only the columns relevant to our analysis

relevant_cols = [
    'track_id',
    'popularity',
    'track_genre',
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'explicit',
    'artists',
    'duration_ms',
    'release_date'
]

cleaned = cleaned[relevant_cols]


In [10]:
# Chech missing values
cleaned.isna().sum().sort_values(ascending=False)

tempo               22114
artists                 1
track_id                0
popularity              0
track_genre             0
danceability            0
energy                  0
loudness                0
speechiness             0
acousticness            0
instrumentalness        0
liveness                0
valence                 0
explicit                0
duration_ms             0
release_date            0
dtype: int64

In [11]:
cleaned["track_genre"].value_counts()

track_genre
acoustic             1000
punk-rock            1000
progressive-house    1000
power-pop            1000
pop                  1000
                     ... 
folk                 1000
emo                  1000
electronic           1000
electro              1000
world-music          1000
Name: count, Length: 114, dtype: int64

In [12]:
# Choose at least 5 genres
selected_genres = ["classical", "hip-hop", "country", "electronic", "metal", "pop"]

cleaned = cleaned[cleaned["track_genre"].isin(selected_genres)]

In [16]:
cleaned['release_date']

16000    2001-12-01
16001    2005-04-15
16002          1984
16003          1972
16004       1987-06
            ...    
81995    2010-11-08
81996    2008-10-08
81997    2018-06-09
81998    2009-08-09
81999    2005-12-15
Name: release_date, Length: 6000, dtype: object

In [17]:
# Convert release date
cleaned['release_date'] = pd.to_datetime(
    cleaned['release_date'].astype(str),
    format='mixed',
    errors = 'coerce'
)

# Extract year
cleaned['release_year'] = cleaned['release_date'].dt.year

In [18]:
cleaned[["release_date", "release_year"]].head()

,release_date,release_year
16000,2001-12-01,2001
16001,2005-04-15,2005
16002,1984-01-01,1984
16003,1972-01-01,1972
16004,1987-06-01,1987


The `release_date` column originally contained inconsistent date formats. To keep the data consistent, we converted the column to datetime format and extracted a new `release_year` feature to support more efficient analysis. 

In [19]:
# Convert duration into minutes
cleaned["duration_min"] = cleaned["duration_ms"] / 60000

In [20]:
cleaned[["duration_ms", "duration_min"]].head()

,duration_ms,duration_min
16000,298266,4.971100
16001,482586,8.043100
16002,219437,3.657283
16003,299146,4.985767
16004,387716,6.461933


We converted `duration_ms` into `duration_min` because minutes are easier to interpret than milliseconds.

In [21]:
cleaned["popularity"].describe()

count    6000.000000
mean       33.908000
std        30.535143
min         0.000000
25%         0.000000
50%        38.000000
75%        63.250000
max       100.000000
Name: popularity, dtype: float64

In [22]:
cleaned["popularity"].quantile([0.5, 0.75, 0.8, 0.9, 0.95])

0.50    38.00
0.75    63.25
0.80    66.00
0.90    72.00
0.95    77.00
Name: popularity, dtype: float64

In [23]:
for cutoff in [50, 55, 60]:
    prop_popular = (cleaned["popularity"] >= cutoff).mean()
    print(cutoff, prop_popular)

50 0.4111666666666667
55 0.37716666666666665
60 0.30883333333333335


In [24]:
popularity_cutoff = 55
cleaned["is_popular"] = cleaned["popularity"] >= popularity_cutoff

We compared popularity cutoffs of 50, 55, and 60. A cutoff of 50 labeled about 41.1% of tracks as popular, while 60 labeled only about 30.9%. We chose 55 because it gives a stricter definition of popularity while still keeping about 37.7% of tracks in the popular class. This makes the classification task more meaningful without making the target too imbalanced.


In [26]:
# Final Look
cleaned.head()

,track_id,popularity,track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,explicit,artists,duration_ms,release_date,release_year,duration_min,is_popular
16000,7wrYBASu0OoxoDErd4Edxd,58,classical,0.643,0.268,-15.073,0.0900,0.593,0.000002,0.316,0.620,NaN,False,Bombay Jayashri,298266,2001-12-01,2001,4.971100,True
16001,72HdutlIHBZJ7WT1xVAAZT,59,classical,0.484,0.898,-4.132,0.1640,0.365,0.000000,0.091,0.680,NaN,False,Shankar;Ehsaan;Loy;Alisha Chinai;Shankar Mahad...,482586,2005-04-15,2005,8.043100,True
16002,7JGgKHHDgJCJkQCQxyHHdl,54,classical,0.608,0.638,-6.008,0.0292,0.581,0.017200,0.448,0.439,140.109,False,Bombay Jayashri;DJ Aftab,219437,1984-01-01,1984,3.657283,False
16003,3YRj4jmwois2ctPnhwSwFo,68,classical,0.695,0.293,-16.278,0.0431,0.596,0.015800,0.132,0.637,NaN,False,Bombay Jayashri,299146,1972-01-01,1972,4.985767,True
16004,3tp3ij9dtY3CacQgd1OvRf,59,classical,0.583,0.308,-18.303,0.0465,0.581,0.010600,0.257,0.241,118.226,False,Bombay Jayashri;Swattrex,387716,1987-06-01,1987,6.461933,True


In [27]:
# After filtering, the dataset contains 6,000 observations and 16 variables.
# This cleaned dataset will be used for the remainder of our analysis and modeling.
cleaned.shape

(6000, 19)

In [69]:
# Preview for the website
preview_cols = [
    'track_id',
    'popularity',
    'track_genre',
    'danceability',
    'acousticness',
    'release_year',
    'tempo',
    'explicit',
    'is_popular'
]

print(cleaned[preview_cols].head().to_markdown(index=False))

| track_id               |   popularity | track_genre   |   danceability |   acousticness |   release_year |   tempo | explicit   | is_popular   |
|:-----------------------|-------------:|:--------------|---------------:|---------------:|---------------:|--------:|:-----------|:-------------|
| 7wrYBASu0OoxoDErd4Edxd |           58 | classical     |          0.643 |          0.593 |           2001 | nan     | False      | True         |
| 72HdutlIHBZJ7WT1xVAAZT |           59 | classical     |          0.484 |          0.365 |           2005 | nan     | False      | True         |
| 7JGgKHHDgJCJkQCQxyHHdl |           54 | classical     |          0.608 |          0.581 |           1984 | 140.109 | False      | False        |
| 3YRj4jmwois2ctPnhwSwFo |           68 | classical     |          0.695 |          0.596 |           1972 | nan     | False      | True         |
| 3tp3ij9dtY3CacQgd1OvRf |           59 | classical     |          0.583 |          0.581 |           1987 | 118.226 |

### 2.1 Univariate Analysis

In [70]:
import plotly.express as px

# Popularity distribution
fig = px.histogram(
    cleaned,
    x='popularity',
    nbins=20,
    title='Popularity Distribution'
)

fig.show()

In [71]:
fig = px.histogram(
    cleaned,
    x="danceability",
    nbins=20,
    title="Danceability Distribution"
)

fig.show()

This histogram above shows the distribution of danceability scores across Spotify tracks. Most tracks have danceability values between 0.4 and 0.8, with the highest concentration around 0.55 to 0.65. This suggests that many songs in the dataset are moderately danceable, while very low-danceability and extremely high-danceability tracks are less common.

In [72]:
fig = px.histogram(
    cleaned,
    x="energy",
    nbins=30,
    title="Distribution of Spotify Track Energy"
)

fig.show()

This histogram shows the distribution of energy scores across Spotify tracks. Most tracks have moderate to high energy values, especially between 0.5 and 1.0, while fewer tracks have very low energy. This suggests that many tracks in the selected genres are relatively active or intense, so energy may be a useful audio feature to include when predicting popularity.

In [73]:
fig = px.histogram(
    cleaned,
    x='explicit',
    title='Explicit vs Non-Explicit Tracks'
)

fig.show()

This chart reveals a significant difference between explicit and non-explicit tracks in the dataset. It appears that non-explicit songs make up the vast majority of tracks, with over 100,000 entries, while explicit songs account for only a much smaller portion of the dataset. This suggests that mainstream music in the dataset is still largely dominated by non-explicit content. This imbalance may also affect later analysis or predictive modeling, where patterns associated with non-explicit tracks could have a stronger influence on the model due to their much larger representation in the dataset.

In [74]:
fig = px.box(
    cleaned,
    y="duration_min",
    title="Distribution of Track Duration"
)

fig.show()

In [75]:
cleaned["duration_min"].describe()

count    6000.000000
mean        3.766399
std         1.856797
min         0.290883
25%         2.926900
50%         3.529192
75%         4.256604
max        44.114433
Name: duration_min, dtype: float64

In [76]:
cleaned["duration_min"].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

0.50    3.529192
0.75    4.256604
0.90    5.288000
0.95    6.417857
0.99    9.381717
Name: duration_min, dtype: float64

In [77]:
duration_cutoff = cleaned["duration_min"].quantile(0.99)

cleaned_duration_filtered = cleaned[
    cleaned["duration_min"] <= duration_cutoff
]

In [78]:
cleaned_duration_filtered["duration_min"].describe()

count    5940.000000
mean        3.655822
std         1.304766
min         0.290883
25%         2.920633
50%         3.520000
75%         4.210775
max         9.381550
Name: duration_min, dtype: float64

We checked numeric summaries and possible outliers for important audio features. We did not automatically remove outliers because unusual values may represent real songs, but checking them helps us understand whether any columns contain extreme or suspicious values.

We also checked the distribution of track duration and found that most tracks are between about 3 and 5 minutes long. The median duration is about 3.55 minutes, and 99% of tracks are under about 8.85 minutes. However, the maximum duration is about 87.29 minutes. After inspecting the longest tracks, we found that many were continuous DJ mixes, sleep sounds, or long recordings, so they appear to be real tracks rather than data errors. To reduce the influence of extreme duration values, we used the 99th percentile as a cutoff for some analyses involving duration.

### 2.2 Bivariate Analysis

In [79]:
fig = px.scatter(
    cleaned,
    x='danceability',
    y='popularity',
    title='Danceability vs Popularity',
    opacity=0.2
)

fig.show()

The scatterplot suggests a weak positive relationship between popularity and danceability. Songs with higher danceability appear to be more popular, however the relationship is not strongly linear. 

In [80]:
fig = px.box(
    cleaned,
    x='explicit',
    y='popularity',
    title='Popularity by Explicit Content'
)

fig.show()

Interestingly, although non-explicit tracks make up the majority of the dataset, explicit tracks appear to have a slightly higher median popularity. This may suggest that explicit content is relatively common among more popular or commercially successful tracks. However, both groups still show a wide spread in popularity, indicating that explicitness alone is not enough to determine whether a song becomes popular.

In [81]:
fig = px.box(
    cleaned,
    x="track_genre",
    y="popularity",
    title="Popularity Distribution by Genre"
)

fig.show()

This box plot compares Spotify popularity scores across selected genres. Pop has the highest median popularity, followed by hip-hop and metal, while classical has the lowest median popularity. This shows that genre may be an important predictor of whether a track is popular, although the wide ranges show that genre alone cannot fully explain popularity.

In [82]:
genre_popularity = (cleaned.groupby('track_genre')['is_popular']
                    .mean()
                    .sort_values(ascending=False))
genre_popularity

track_genre
pop           0.644
hip-hop       0.571
metal         0.532
electronic    0.310
country       0.157
classical     0.049
Name: is_popular, dtype: float64

In [83]:
fig = px.box(
    cleaned,
    x="track_genre",
    y="danceability",
    title="Danceability Distribution by Genre"
)

fig.show()


This box plot shows how danceability varies across the selected genres. Hip-hop has the highest typical danceability, while classical tends to have the lowest. Pop and electronic are also generally more danceable. Which mean that danceability is different across genres, so audio features like this could be useful when comparing or predicting track popularity.

### Energy distribution

In [84]:
fig = px.scatter(
    cleaned,
    x="energy",
    y="popularity",
    color="track_genre",
    opacity=0.5,
    title="Energy vs. Popularity by Genre"
)

fig.show()

This scatter plot looks at the relationship between energy and popularity across the selected genres. Overall, higher energy does not always mean higher popularity, since songs with similar energy levels can have very different popularity scores. Still, the genre colors show some patterns. Pop, hip-hop, and metal tracks appear more often in the higher popularity range, while classical tracks tend to have lower energy and lower popularity. This suggests that energy by itself is not enough to explain popularity, but it could still be useful when combined with genre and other audio features.

## 2.3 Interesting aggregates

In [85]:
#How selected genres differ in popularity and audio features

genre_summary = (
    cleaned
    .groupby("track_genre")
    .agg(
        num_tracks=("track_id", "count"),
        avg_popularity=("popularity", "mean"),
        median_popularity=("popularity", "median"),
        popular_rate=("is_popular", "mean"),
        avg_danceability=("danceability", "mean"),
        avg_energy=("energy", "mean"),
        avg_valence=("valence", "mean"),
        avg_acousticness=("acousticness", "mean"),
        avg_instrumentalness=("instrumentalness", "mean"),
        avg_tempo=("tempo", "mean")
    )
    .reset_index()
    .sort_values("avg_popularity", ascending=False)
)

genre_summary.round(3)

,track_genre,num_tracks,avg_popularity,median_popularity,popular_rate,avg_danceability,avg_energy,avg_valence,avg_acousticness,avg_instrumentalness,avg_tempo
5,pop,1000,47.576,66.0,0.644,0.630,0.606,0.506,0.344,0.009,122.053
2,electronic,1000,44.325,48.0,0.310,0.653,0.695,0.392,0.177,0.249,123.788
4,metal,1000,43.705,57.0,0.532,0.464,0.840,0.418,0.037,0.065,128.060
3,hip-hop,1000,37.759,58.0,0.571,0.736,0.683,0.551,0.194,0.011,117.119
1,country,1000,17.028,0.0,0.157,0.555,0.597,0.521,0.321,0.006,125.018
0,classical,1000,13.055,3.0,0.049,0.382,0.190,0.381,0.920,0.619,108.433


This table gives a quick summary of how the selected genres differ in popularity and audio features. Pop has the highest popular rate, with about 64% of pop tracks labeled as popular. Hip-hop and metal also have relatively high popular rates, while classical has the lowest. The audio features also show clear genre differences. Hip-hop has the highest average danceability, metal has the highest average energy, and classical has the highest acousticness and instrumentalness but much lower energy. Overall, the table suggests that genre is related to both popularity and the sound of a track, so using genre together with audio features could be helpful for predicting whether a song is popular.

In [86]:
# How do audio features differ between popular and non-popular songs

audio_feature_means = (
    cleaned.groupby('is_popular')[
        [
            'danceability',
            'energy',
            'loudness',
            'speechiness',
            'acousticness',
            'instrumentalness',
            'liveness',
            'valence',
            'tempo'
        ]
    ]
    .mean()
)

pd.set_option('display.max_columns', None)
audio_feature_means

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
is_popular,,,,,,,,,
False,0.549331,0.551767,-10.070876,0.068770,0.387451,0.211277,0.183513,0.457427,121.484300
True,0.604593,0.684344,-6.759222,0.087698,0.240986,0.074451,0.179019,0.468569,120.925376


The table compares the average audio features between popular and non-popular songs. Popular songs tend to have slightly higher average danceability scores and higher loudness values, suggesting that more rhythmically engaging and louder tracks may perform better on Spotify. Popular songs also show lower average acousticness and instrumentalness, indicating that mainstream songs in the dataset are generally less acoustic and more vocal-focused.

However, many of the differences between the two groups remain relatively small, especially for features such as energy, speechiness, and valence. This suggests that no single audio feature alone strongly determines popularity, and that song popularity is likely influenced by a combination of multiple audio characteristics and genre.

## 3. Assessment of Missingness

In [87]:
missing_summary = (
    cleaned.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("num_missing")
)

missing_summary["percent_missing"] = missing_summary["num_missing"] / len(cleaned)
missing_summary

,num_missing,percent_missing
tempo,1205,0.200833
track_id,0,0.000000
popularity,0,0.000000
is_popular,0,0.000000
duration_min,0,0.000000
release_year,0,0.000000
release_date,0,0.000000
duration_ms,0,0.000000
artists,0,0.000000
explicit,0,0.000000


In [88]:
cleaned["tempo_missing"] = cleaned["tempo"].isna()

In [89]:
cleaned["tempo_missing"].value_counts(normalize=True)

tempo_missing
False    0.799167
True     0.200833
Name: proportion, dtype: float64

In [90]:
observed_ks = stats.ks_2samp(
    cleaned.loc[cleaned["tempo_missing"], "popularity"].dropna(),
    cleaned.loc[~cleaned["tempo_missing"], "popularity"].dropna()
).statistic

observed_ks

np.float64(0.0928076012789948)

### Part A: Test if tempo missingness depends on release year

Null Hypothesis: The `tempo_missingness` does not depend on `release_year`. 
<br>
Alternative Hypothesis: The `tempo_missingness` depends on `release_year`. 

Test Statistic: Absolute Difference in Means

In [91]:
# Observed Statistic
observed_diff = abs(
    cleaned
    .groupby('tempo_missing')['release_year']
    .mean()
    .diff()
    .iloc[-1]
)

observed_diff

np.float64(4.59543611732488)

In [92]:
# Permutation Test
diffs = []

for _ in range(5000):

    shuffled = cleaned.copy()

    shuffled['tempo_missing'] = np.random.permutation(
        shuffled['tempo_missing']
    )

    diff = abs(
        shuffled
        .groupby('tempo_missing')['release_year']
        .mean()
        .diff()
        .iloc[-1]
    )

    diffs.append(diff)

diffs = np.array(diffs)

# Compute P-value
p_value = np.mean(
    diffs >= observed_diff
)

p_value

np.float64(0.0)

In [93]:
# Visualization
fig = px.histogram(
    x=np.append(diffs, observed_diff),
    nbins=30,
    title=f'Permutation Distribution (p = {p_value})'
)

fig.add_vline(
    x=observed_diff,
    line_color='red',
    line_width=3,
    annotation_text='Observed Difference'
)

fig.show()

The permutation test produced a p-value of approximately 0. Since this is below 0.05, we reject the null hypothesis. This suggests that whether a track is missing a tempo value depends on the track’s release year.

### Part B: Test if tempo missingness depends on track_genre

Null Hypothesis: The `tempo_missingness` does not depend on `track_genre`. 
<br>
Alternative Hypothesis: The `tempo_missingness` depends on `track_genre`. 

Test Statistic: Total Variation Distance (TVD)

In [94]:
def tvd(data, group_col, cat_col):

    props = (
        data
        .pivot_table(
            index=cat_col,
            columns=group_col,
            aggfunc='size',
            fill_value=0
        )
    )

    props = props / props.sum()

    return 0.5 * np.abs(
        props[True] - props[False]
    ).sum()


# Compute Observed TVD
observed_tvd = tvd(
    cleaned,
    'tempo_missing',
    'track_genre'
)

observed_tvd

np.float64(0.17168644724146437)

In [95]:
observed_tvd = tvd(cleaned, "tempo_missing", "track_genre")
observed_tvd

np.float64(0.17168644724146437)

In [96]:
# Permutation Test
tvds = []

for _ in range(5000):

    shuffled = cleaned.copy()

    shuffled['tempo_missing'] = np.random.permutation(
        shuffled['tempo_missing']
    )

    tvds.append(
        tvd(
            shuffled,
            'tempo_missing',
            'track_genre'
        )
    )

tvds = np.array(tvds)

# Compute P-value
p_value = np.mean(tvds >= observed_tvd)

p_value

np.float64(0.0)

In [97]:
# Visualization
fig = px.histogram(
    x=tvds,
    nbins=30,
    title=f'Permutation Distribution (p = {p_value})'
)

fig.add_vline(
    x=observed_tvd,
    line_color='red',
    line_width=3,
    annotation_text='Observed TVD'
)

fig.show()

Since the observed TVD lies far outside the permuatation and the p-value is approximately 0, therefore we reject the null hypothesis and conclude that the missingness of `tempo` depends on `track_genre`.

In [98]:
numeric_cols_to_test = [
    "popularity",
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "duration_min",
    "loudness"
]

results = []

for col in numeric_cols_to_test:
    observed = stats.ks_2samp(
        cleaned.loc[cleaned["tempo_missing"], col].dropna(),
        cleaned.loc[~cleaned["tempo_missing"], col].dropna()
    ).statistic
    
    simulated = []
    
    for _ in range(1000):
        shuffled = cleaned.copy()
        shuffled[col] = np.random.permutation(cleaned[col])
        
        groups = shuffled.groupby("tempo_missing")[col]
        
        stat = stats.ks_2samp(
            groups.get_group(True).dropna(),
            groups.get_group(False).dropna()
        ).statistic
        
        simulated.append(stat)
    
    p_value = np.mean(np.array(simulated) >= observed)
    
    results.append({
        "column": col,
        "observed_ks": observed,
        "p_value": p_value
    })

missingness_results = pd.DataFrame(results).sort_values("p_value")
missingness_results

,column,observed_ks,p_value
0,popularity,0.092808,0.000
1,danceability,0.092850,0.000
2,energy,0.273084,0.000
3,valence,0.103782,0.000
4,acousticness,0.218579,0.000
5,instrumentalness,0.130663,0.000
7,speechiness,0.121360,0.000
8,duration_min,0.092064,0.000
9,loudness,0.222741,0.000
6,liveness,0.057373,0.001


In [99]:
def tvd(data, group_col, cat_col):
    props = (
        data
        .pivot_table(index=cat_col, columns=group_col, aggfunc="size", fill_value=0)
    )
    
    props = props / props.sum()
    
    return 0.5 * np.abs(props[True] - props[False]).sum()

In [100]:
categorical_cols_to_test = ["explicit", "track_genre", "is_popular"]

cat_results = []

for col in categorical_cols_to_test:
    observed = tvd(cleaned, "tempo_missing", col)
    
    simulated = []
    
    for _ in range(1000):
        shuffled = cleaned.copy()
        shuffled[col] = np.random.permutation(cleaned[col])
        
        stat = tvd(shuffled, "tempo_missing", col)
        simulated.append(stat)
    
    p_value = np.mean(np.array(simulated) >= observed)
    
    cat_results.append({
        "column": col,
        "observed_tvd": observed,
        "p_value": p_value
    })

cat_missingness_results = pd.DataFrame(cat_results).sort_values("p_value")
cat_missingness_results

,column,observed_tvd,p_value
1,track_genre,0.171686,0.000
2,is_popular,0.064887,0.000
0,explicit,0.027592,0.008


We tested several numeric columns and found that tempo missingness depends on many of them, including popularity, danceability, energy, valence, acousticness, instrumentalness, speechiness, duration, and loudness. We chose to discuss energy because it had the largest KS statistic, meaning the energy distribution differed the most between tracks with missing tempo and tracks with non-missing tempo. We also tested categorical columns using total variation distance. 

Overall, tempo missingness does not seem to be MCAR, because it depends on observed features such as energy. It is more likely MAR, since the missingness can be partially explained by other observed columns in the dataset. We cannot conclude that it is NMAR from the data alone, because that would require knowing whether missingness depends on the unobserved tempo values themselves.

## 4. Hypothesis Testing 

**Null Hypothesis**: The distribution of `track_genre` is the same for popular and non-popular songs.
<br><br>
**Alternative Hypothesis**: The distribution of `track_genre` differs between popular and non-popular songs.
<br><br>
**Test Statistic**: TVD between the genre distributions of popular and non-popular songs.

In [101]:
# Observed TVD
observed_tvd = tvd(
    cleaned,
    'is_popular',
    'track_genre'
)

observed_tvd

np.float64(0.436688400182054)

In [102]:
# Permutation Test
tvds = []

for _ in range(1000):

    shuffled = cleaned.copy()

    shuffled['is_popular'] = np.random.permutation(
        shuffled['is_popular']
    )

    tvds.append(
        tvd(
            shuffled,
            'is_popular',
            'track_genre'
        )
    )

tvds = np.array(tvds)

# P-value
p_value = np.mean(
    tvds >= observed_tvd
)

p_value

np.float64(0.0)

In [103]:
# Visualization
fig = px.histogram(
    x=tvds,
    nbins=30,
    title=f'Permutation Distribution (p = {p_value})'
)

fig.add_vline(
    x=observed_tvd,
    line_color='red',
    line_width=3,
    annotation_text='Observed TVD'
)

fig.show()

We used a permutation test to test whether popularity differs across the selected genres. The p-value was approximately 0, so I reject the null hypothesis at the 0.05 significance level. This provides evidence that genre is associated with track popularity in this dataset.


## 5. Framing a Prediction Problem

**Prediction Problem:**
Can we predict whether a Spotify track is popular using its audio features, genre, and track metadata?

**Type:** Binary Classification — we predict whether a track is popular (1) or not popular (0).

**Response Variable:** `is_popular` — created by thresholding `popularity` at 55. Tracks scoring 55 or above are labeled popular (1), the rest not popular (0). This labels the top 37.7% of tracks as popular.

**Features known at time of prediction:**
- Audio features: danceability, energy, loudness, acousticness, instrumentalness, speechiness, liveness, valence, tempo
- Genre: track_genre
- Metadata: duration_min, explicit, release_year, num_artists

**Evaluation Metric:** F1 score — because the dataset is imbalanced (only ~37% of tracks are popular). Accuracy would be misleading since a model could achieve high accuracy by simply predicting "not popular" for every track. F1 balances precision and recall, giving a more honest measure of performance.

## 6. Baseline Model 

In [104]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_features = ["danceability", "track_genre"]

X_base = cleaned[baseline_features]
y = cleaned["is_popular"]

X_train, X_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)

# 3. Define transformers 
categorical = X_base.select_dtypes(include=['object', 'category']).columns 
numerical = X_base.select_dtypes(include=['number', 'bool']).columns

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

# 4. Build pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# 5. Fit and evaluate
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, pos_label=True))


Accuracy: 0.7391666666666666
Precision: 0.6855670103092784
Recall: 0.5820568927789934
F1: 0.6295857988165681


The baseline model uses only 2 features:
- danceability (quantitative) — scaled with StandardScaler
- track_genre (nominal) — encoded with OneHotEncoder

This intentionally minimal model establishes a performance floor.
F1 score of 0.630 shows significant room for improvement.

## 7. Final Model

### Engineer new features

In [105]:
cleaned['num_artists'] = (
    cleaned['artists']
    .str.split(';')
    .str.len()
)
#cleaned["loudness_energy_ratio"] = cleaned['loudness'] / cleaned['energy']
cleaned["tempo_filled"] = cleaned["tempo"].fillna(
    cleaned.groupby("track_genre")["tempo"].transform("median")
)
# Avoid division by zero when creating ratio
cleaned["loudness_energy_ratio"] = cleaned["loudness"] / cleaned["energy"].replace(0, np.nan)


In [106]:
audio_features = ["danceability", "energy", "loudness", "acousticness", 
                  "instrumentalness", "speechiness", "liveness", "valence", 
                  "tempo_filled", "duration_min", "release_year", "num_artists",
                  "loudness_energy_ratio"]

corr_with_target = cleaned[audio_features + ['is_popular']].corr()['is_popular'].sort_values()
print(corr_with_target)

instrumentalness        -0.209486
acousticness            -0.199259
num_artists             -0.038547
liveness                -0.015294
tempo_filled             0.009204
valence                  0.022544
duration_min             0.041544
loudness_energy_ratio    0.069503
speechiness              0.124142
danceability             0.152616
energy                   0.241286
loudness                 0.250726
release_year             0.295150
is_popular               1.000000
Name: is_popular, dtype: float64


In [107]:
corr_plot = corr_with_target.drop('is_popular')

fig2 = px.bar(
    x=corr_plot.values,
    y=corr_plot.index,
    orientation='h',
    title='Feature Correlation with is_popular',
    labels={'x': 'Correlation', 'y': 'Feature'},
    color=corr_plot.values,
    color_continuous_scale='RdBu'
)

fig2.show()

Not all features relate to popularity in the same way. Features like 
`release_year`, `loudness`, and `energy` show positive correlations: songs 
that are louder, more energetic, or more recently released tend to score 
higher in popularity. On the other side, `instrumentalness` and `acousticness` 
are negatively correlated: songs that are heavily instrumental or acoustic 
(think classical or folk) tend to be less popular on Spotify's mainstream 
charts. It's worth noting that correlation only captures linear relationships, 
so some features may still be useful even if their correlation looks small.

In [108]:
from sklearn.model_selection import GridSearchCV

final_features = ["danceability", "acousticness", "energy", "loudness", "instrumentalness",
                  "speechiness", "valence", "duration_min", "explicit", "track_genre", 
                  "loudness_energy_ratio", "tempo_filled", "release_year", "num_artists"]

X_final = cleaned[final_features]
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final, y, test_size=0.2, random_state=42)


# Pipeline (same as before)
categorical = X_final.select_dtypes(include=['object', 'category']).columns.tolist() 
numerical = X_final.select_dtypes(include=['number', 'bool']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# 5. GridSearchCV - fill in the hyperparameters to try!
param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [5, 10, 20],
}

grid_search = GridSearchCV(final_pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(X_train_final, y_train_final)

print("Best params:", grid_search.best_params_)
y_pred_final = grid_search.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:", recall_score(y_test_final, y_pred_final))
print("F1:", f1_score(y_test_final, y_pred_final, pos_label=True))


Best params: {'classifier__max_depth': 20, 'classifier__n_estimators': 200}
Accuracy: 0.82
Precision: 0.806615776081425
Recall: 0.6936542669584245
F1: 0.7458823529411764


In [109]:
feature_names = (numerical + categorical)  # or however you defined them
importances = grid_search.best_estimator_.named_steps['classifier'].feature_importances_

# For just the numeric features before one-hot encoding
importance_df = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': grid_search.best_estimator_.named_steps['classifier'].feature_importances_[:len(X_train_final.columns)]
}).sort_values('importance', ascending=False)

print(importance_df)

                  feature  importance
1            acousticness    0.098025
11           tempo_filled    0.086297
2                  energy    0.081412
9             track_genre    0.078082
7            duration_min    0.074249
0            danceability    0.071554
6                 valence    0.071319
3                loudness    0.070298
5             speechiness    0.070206
10  loudness_energy_ratio    0.066643
4        instrumentalness    0.055309
13            num_artists    0.029442
12           release_year    0.018915
8                explicit    0.013801


Note: Tree-based feature importance can underestimate features that work 
together with others. For example, `release_year` ranks low here (0.019) 
but shows high permutation importance (0.035), suggesting it contributes 
meaningfully when combined with other features. This is why we use 
permutation importance as our primary measure of feature contribution.

### Permutation Importance

In [110]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    grid_search.best_estimator_, 
    X_test_final, 
    y_test_final, 
    n_repeats=10,
    scoring='f1',
    random_state=42
)

perm_df = pd.DataFrame({
    'feature': X_test_final.columns,
    'importance': result.importances_mean
}).sort_values('importance', ascending=False)

print(perm_df)

                  feature  importance
9             track_genre    0.281631
12           release_year    0.039541
1            acousticness    0.033570
7            duration_min    0.027261
3                loudness    0.022331
0            danceability    0.020754
4        instrumentalness    0.018505
2                  energy    0.018481
8                explicit    0.016196
6                 valence    0.016151
10  loudness_energy_ratio    0.014377
13            num_artists    0.011565
11           tempo_filled    0.004730
5             speechiness    0.004441


In [111]:
fig3 = px.bar(
    perm_df.sort_values('importance'),
    x='importance',
    y='feature',
    orientation='h',
    title='Permutation Importance — Final Model',
    labels={'importance': 'Mean F1 Drop When Shuffled', 'feature': 'Feature'},
    color='importance',
    color_continuous_scale='RdYlGn'

)

fig3.update_xaxes(type='log')
fig3.show()

### Feature Exclusion: liveness
We tested adding `liveness` to the final model since it is one of Spotify's 
standard audio features. However:
- Its correlation with `is_popular` was nearly zero (-0.015)
- Its permutation importance was among the lowest (0.005)
- Adding it to the model hurt overall F1 performance

Interestingly, removing `liveness` caused `speechiness` to improve from 
negative to positive permutation importance. This suggests the two features 
were correlated and interfering with each other — a classic sign of 
multicollinearity. Features like `acousticness` and `energy` likely already 
capture similar information about a track's recording environment, making 
`liveness` redundant.

We therefore excluded `liveness` from our final feature set.

### Confusion Matrix

In [112]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

# Compute confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final)

# Plot using plotly
labels = ['Not Popular', 'Popular']
fig1 = ff.create_annotated_heatmap(
    cm,
    x=labels,
    y=labels,
    colorscale='Blues',
    showscale=True
)

fig1.update_layout(
    title='Confusion Matrix — Final Model',
    xaxis_title='Predicted',
    yaxis_title='Actual'
)

fig1.show()

## 8. Fairness Analysis

#### Does Our Model Perform Equally Well Across Genre Groups?

**Evaluation metric**: F1 score

**Groups**:
- Group X: Lower-popularity genres — classical, country
- Group Y: Higher-popularity genres — electronic, hip-hop, metal, pop

**Null hypothesis**: The model is fair. Its F1 score for lower-popularity and 
higher-popularity genres are roughly the same, and any differences are due 
to random chance.

**Alternative hypothesis**: The model is unfair. Its F1 score for 
lower-popularity genres is lower than for higher-popularity genres.

**Test statistic**: Difference in F1 scores (lower genres minus higher genres)

**Significance level**: 0.05

In [113]:
lower_genres = ["classical", "country"]
higher_genres = ["electronic", "hip-hop", "metal", "pop"]
lower_mask = X_test_final['track_genre'].isin(lower_genres)
higher_mask = X_test_final['track_genre'].isin(higher_genres)


f1_lower_genres = f1_score(y_test[lower_mask], y_pred_final[lower_mask], pos_label=True)
f1_higer_genres = f1_score(y_test[higher_mask], y_pred_final[higher_mask], pos_label=True)

observed_diff = f1_lower_genres - f1_higer_genres
print("Observed difference:", observed_diff)

Observed difference: -0.6808461978273299


In [114]:
import numpy as np

n_simulations = 1000
simulated_diffs = []

for _ in range(n_simulations):
    # Shuffle the explicit labels
    shuffled_mask = np.random.permutation(lower_mask)

    f1_shuffled_lower = f1_score(y_test[shuffled_mask], y_pred_final[shuffled_mask], pos_label=True)
    f1_shuffled_higher = f1_score(y_test[~shuffled_mask], y_pred_final[~shuffled_mask], pos_label=True)

    simulated_diffs.append(f1_shuffled_lower - f1_shuffled_higher)

# Calculate p-value
p_value = np.mean(np.array(simulated_diffs) <= observed_diff)
print("P-value:", p_value)

P-value: 0.0


In [115]:
fig4 = px.histogram(
    x=simulated_diffs,
    nbins=30,
    title='Fairness Permutation Test — Lower vs Higher Popularity Genres',
    labels={'x': 'Difference in F1 Score (Lower - Higher Genres)'}
)

fig4.add_vline(
    x=observed_diff,
    line_dash='dash',
    annotation_text=f'Observed: {observed_diff:.3f}'
)

fig4.show()

**p-value: 0.0** — Since 0.0 < 0.05, we **reject the null hypothesis**.

The plot above shows that the observed difference of -0.681 falls far outside 
the distribution of simulated differences, confirming this result is not due 
to random chance. Our model performs significantly worse on classical and 
country tracks than on pop, hip-hop, metal, and electronic tracks. This is 
likely due to class imbalance — lower-popularity genres have very few popular 
tracks in the training data (classical at 4.9%, country at 15.7%), making it 
harder for the model to learn what makes them popular compared to genres like 
pop (64.4%).

#### Is Our Model Fair Across Explicit and Non-Explicit Tracks?
**Evaluation metric**: F1 score

**Groups**:
- Group X: Explicit tracks (`explicit=True`)
- Group Y: Non-explicit tracks (`explicit=False`)

**Null hypothesis**: The model is fair. Its F1 score for explicit and 
non-explicit tracks are roughly the same, and any differences are due 
to random chance.

**Alternative hypothesis**: The model is unfair. Its F1 score for explicit 
tracks is lower than for non-explicit tracks.

**Test statistic**: Difference in F1 scores (explicit minus non-explicit)

**Significance level**: 0.05

In [116]:
# Step 1: Calculate observed statistic
explicit_mask = X_test_final['explicit'] == True
non_explicit_mask = X_test_final['explicit'] == False

f1_explicit = f1_score(y_test_final[explicit_mask], y_pred_final[explicit_mask], pos_label=True)
f1_non_explicit = f1_score(y_test_final[non_explicit_mask], y_pred_final[non_explicit_mask], pos_label=True)

observed_diff = f1_explicit - f1_non_explicit
print("Observed difference:", observed_diff)


Observed difference: -0.016505447365804393


In [117]:
# Step 2: Permutation test
n_simulations = 1000
simulated_diffs = []

for _ in range(n_simulations):
    shuffled_explicit = np.random.permutation(explicit_mask)
    
    f1_shuffled_explicit = f1_score(y_test_final[shuffled_explicit], y_pred_final[shuffled_explicit], pos_label=True)
    f1_shuffled_non_explicit = f1_score(y_test_final[~shuffled_explicit], y_pred_final[~shuffled_explicit], pos_label=True)
    
    simulated_diffs.append(f1_shuffled_explicit - f1_shuffled_non_explicit)

# Step 3: Calculate p-value
p_value = np.mean(np.array(simulated_diffs) <= observed_diff)
print("P-value:", p_value)


P-value: 0.401


In [118]:
# Step 4: Visualization
fig5 = px.histogram(
    x=simulated_diffs,
    nbins=30,
    title='Fairness Permutation Test — Explicit vs Non-Explicit Tracks',
    labels={'x': 'Difference in F1 Score (Explicit - Non-Explicit)'}
)

fig5.add_vline(
    x=observed_diff,
    line_dash='dash',
    annotation_text=f'Observed: {observed_diff:.3f}'
)

fig5.show()

Based on our permutation test, we obtained a p-value of 0.384. Since this is greater than our significance level of 0.05, we fail to reject the null hypothesis. The observed F1 difference of -0.01 between explicit and non-explicit tracks is likely due to randomness.

